# Mapa multibloco historico

**Papel nesta entrega:** Suplementar: analise historica descrita no trabalho

**Origem:** https://colab.research.google.com/drive/1d5LPDDE21BdRxAM_o1FueqvMfKgW0zjr

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Dependencias e caminhos sao os da campanha historica, nao das quatro execucoes padronizadas. Esses cadernos nao devem substituir os resultados principais. Consulte o mapa de cobertura e mantenha separadas as datas de execucao. 

# TCC | Etapa 03 - Mapa fenotipico multibloco em R1-R3
**Mario Costa Cruz | IMTM HepG2 | EUbOPEN/EU-OS bioactives**

A caracterizacao dos 12 candidatos mostrou que sinal, magnitude e composicao do perfil sao dimensoes distintas. Agora a unidade de observacao volta a ser **toda a base de tratamentos do desenvolvimento**, nao apenas a lista mitocondrial selecionada.

**Objetivo:** descrever os sete blocos, suas combinacoes de dois e tres blocos e a variabilidade entre replicas/placas. Duas PCA exploratorias (quatro canais e sete blocos) complementam o mapa sem criar rotulos supervisionados.

**Escopo:** pos-hoc, restrito a R1-R3. Nao le nenhum arquivo de R4, nao recalibra P95, nao refaz a normalizacao/selecao e nao muda os 12 candidatos. Os escores sao reutilizados; nao e uma nova validacao independente.

**Execucao:** Ambiente de execucao > Executar tudo. Autorize o Drive. Caminhos e hashes estao configurados. Nao e necessario reexecutar os notebooks anteriores.

**Verificacao antes da entrega:** funcoes, limites, chaves, casos ausentes, simultaneidade, PCA e geracao/exportacao de 10 figuras testados com exemplos artificiais separados; resumo original de 2.465 condicoes conferido. A leitura integral do Parquet e os resultados globais desta etapa ainda precisam ser executados no Colab. Nao existem resultados globais preenchidos manualmente.

## 1. Ambiente e fontes
O notebook usa os arquivos originais de escores por condicao/replica, resumo de tratamentos e referencias. Uma divergencia de hash interrompe a execucao. As saidas sao gravadas em pasta nova, nunca nas fontes.

In [ ]:
from pathlib import Path
import os, sys, importlib.util, subprocess
if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
if importlib.util.find_spec('polars') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'polars>=1.0,<2'])
import polars as pl
from IPython.display import display, Markdown
RAIZ = Path(os.environ.get('TCC_RAIZ', '/content/drive/MyDrive/mba_tcc'))
ES_REL = Path('pipeline_IMTM_HepG2/20260919T204105_625149Z/normalizacao_20260919T205516_579360Z/feature_selection_20260919T211235_656816Z/escores_exploratorios_20260919T230543_380473Z')
HASHES = {'escores': 'fe7a80580f166a10ebe574a6721301e4ad8132b041faf88bc977dda142b9ae61', 'resumo': 'a0a3e1ac3d6ef83673b087439835c830856044a7b72999b9494a8f198d8c60fe', 'referencias': '05ea6446566459559a376078d14c7ec5ef98821e4d34bab6c788ac8ebed6d952'}
NOMES = {'escores': 'escores_condicao_replica.parquet', 'resumo': 'resumo_tratamentos.csv', 'referencias': 'referencias_dmso.csv'}


## 2. Regras de leitura e identificacao
A chave preserva placa, codigo do composto e concentracao. Converte apenas representacoes numericas equivalentes da mesma dose (10 e 10.0), sem supor unidades. Uma condicao precisa das tres replicas avaliaveis e dos sete escores finitos. Ausencia nao sera convertida em ausencia de efeito.

In [ ]:
"""Etapa 03: mapa descritivo multibloco. Nao le R4 nem treina classificadores."""
from pathlib import Path
from decimal import Decimal, InvalidOperation
from itertools import combinations
from collections import Counter
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P, E, D, R = 'Metadata_Plate_ID', 'Metadata_EOS', 'Metadata_Concentration', 'Metadata_Replicate'
KEY = [P, E, '_dose_key']
REPS = ['R1', 'R2', 'R3']
BLOCKS = ['AGP', 'DNA', 'ER', 'Mito', 'Morfologia_Cells', 'Morfologia_Cyto', 'Morfologia_Nuc']
LABELS = ['AGP', 'DNA', 'ER', 'Mito', 'Forma Cells', 'Forma Cyto', 'Forma Nuc']
COLS = ['A_'+b for b in BLOCKS]
CUT = 1.0


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()


def dose_key(x):
    if pd.isna(x) or str(x).strip() == '':
        raise ValueError('Concentracao ausente. Nao e permitido juntar doses distintas.')
    s = str(x).strip()
    try:
        d = Decimal(s)
    except InvalidOperation:
        return 'txt:'+s
    if not d.is_finite():
        raise ValueError('Concentracao nao finita.')
    return 'num:'+format((Decimal(0) if d == 0 else d).normalize(), 'f')


def keys(df):
    df = df.copy()
    for c in [P, E, D]:
        if c not in df or df[c].isna().any():
            raise ValueError('Chave ausente ou nula: '+c)
    for c in [P,E]:
        df[c] = df[c].astype(str).str.strip()
        if df[c].eq('').any(): raise ValueError('Chave vazia: '+c)
    df['_dose_key'] = df[D].map(dose_key)
    return df


def bool_column(x):
    out = x.astype(str).str.strip().str.lower().map({'true':True, 'false':False, '1':True, '0':False})
    if out.isna().any(): raise ValueError('Coluna booleana ausente ou invalida.')
    return out.astype(bool)


def pattern(mask):
    return ' + '.join(b for b,yes in zip(BLOCKS, mask) if yes) or 'Nenhum bloco acima do limite'




In [ ]:
FONTES = {k: RAIZ/ES_REL/nome for k,nome in NOMES.items()}
for key,path in FONTES.items():
    if not path.is_file(): raise FileNotFoundError(f'Fonte original nao localizada: {path}')
    observed = sha256(path)
    if observed != HASHES[key]:
        raise RuntimeError(f'Arquivo diferente da fonte conferida: {path.name}. Nenhuma substituicao automatica sera feita.')
# Polars le o Parquet; conversao limitada aos escores, sem carregar 735 features.
source = pl.read_parquet(FONTES['escores'])
rep = pd.DataFrame(source.to_dict(as_series=False))
original = pd.read_csv(FONTES['resumo'])
references = pd.read_csv(FONTES['referencias'])
if len(original) != 2465 or int(bool_column(original['prioridade_exploratoria']).sum()) != 12:
    raise RuntimeError('Resumo original inesperado; verificar fonte antes de prosseguir.')
print('Fonte de desenvolvimento:', source.shape)
print('Replicas presentes:', sorted(rep[R].dropna().unique()))
print('Limites P95 originais preservados; nenhuma referencia recalculada.')
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = RAIZ/'resultados_tcc'/'mapa_multibloco'/('R1_R3_'+RUN)
OUT.mkdir(parents=True, exist_ok=False)
provenance = dict(execucao=RUN, fontes={k:str(v) for k,v in FONTES.items()}, sha256_fontes=HASHES,
                  criterio='A>1 em pelo menos 2 de 3 replicas',
                  combinacoes='Pares e trios inclusivos; excedencia simultanea nas mesmas replicas',
                  natureza='Descritiva pos-hoc; nao confirmatoria', polars=pl.__version__)
(OUT/'plano_descritivo.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')


## 3. Tres descricoes que nao devem ser confundidas
**Excedencia individual:** um bloco apresenta A>1 em pelo menos duas replicas.

**Consenso marginal:** quais blocos satisfazem essa regra individual. Dois blocos no consenso podem ter excedido juntos em apenas uma replica.

**Excedencia simultanea:** ambos os blocos de um par, ou todos os tres de um trio, apresentam A>1 nas **mesmas** pelo menos duas replicas. A contagem e inclusiva: outros blocos podem exceder tambem. Nao mede sinergia, interacao ou causalidade.

**Padrao exato recorrente:** a mesma configuracao de excedencia dos sete blocos aparece em pelo menos duas replicas. Estas categorias sao exclusivas. Sem repeticao da configuracao completa, a condicao fica como padrao variavel.

A>1 e apenas o limite operacional original. Nenhuma dessas novas descricoes e uma classe biologica validada ou um teste com FDR controlada. A regra original das 12 prioridades, incluindo seu cosseno, e reconstituida apenas para conferi-la.

In [ ]:
def analyze(rep, original):
    """Usa apenas escores por condicao/replica ja calculados no desenvolvimento."""
    rep, original = keys(rep), keys(original)
    for c in [R, 'avaliavel', 'n_pocos', *COLS]:
        if c not in rep: raise ValueError('Esquema incompativel; falta '+c)
    if rep.empty: raise ValueError('Base de escores vazia.')
    if not set(rep[R]).issubset(set(REPS)):
        raise ValueError('Fonte contem replica diferente de R1-R3. R4 NAO sera lida.')
    if rep[E].str.upper().eq('DMSO').any():
        raise ValueError('Esperada tabela de tratamentos, nao de controles DMSO.')
    if rep.duplicated(KEY+[R]).any() or original.duplicated(KEY).any():
        raise ValueError('Chaves duplicadas. Nao agregar silenciosamente.')
    rep['avaliavel'] = bool_column(rep['avaliavel'])
    original['prioridade_exploratoria'] = bool_column(original['prioridade_exploratoria'])
    rep[COLS] = rep[COLS].apply(pd.to_numeric, errors='raise')
    if (rep[COLS] < 0).any().any(): raise ValueError('Escores RMS relativos negativos.')
    n = pd.to_numeric(rep['n_pocos'], errors='raise')
    if (~np.isfinite(n) | (n < 1) | (n != np.floor(n))).any():
        raise ValueError('Contagem de pocos invalida.')
    rep['n_pocos'] = n.astype(int)
    rows, arrays = [], []
    for key, g in rep.groupby(KEY, sort=True, dropna=False):
        g = g.set_index(R).reindex(REPS)
        x = g[COLS].to_numpy(float)
        ok = bool(g['avaliavel'].fillna(False).all() and np.isfinite(x).all())
        row = dict(zip(KEY,key)); row[D] = g[D].dropna().iloc[0]
        row.update(n_replicas_presentes=int(g['n_pocos'].notna().sum()),
                   n_pocos=int(g['n_pocos'].sum()), completa_3rep_7blocos=ok)
        for j,b in enumerate(BLOCKS):
            for k,r in enumerate(REPS): row[f'A_{b}_{r}'] = x[k,j]
            row[f'A_{b}_mediana'] = float(np.median(x[:,j])) if ok else np.nan
            row[f'nrep_{b}_acima'] = int((x[:,j]>CUT).sum()) if ok else np.nan
        if ok:
            active = x>CUT
            consensus = active.sum(axis=0)>=2
            codes = [sum(int(v)<<j for j,v in enumerate(a)) for a in active]
            counts = Counter(codes); mode, freq = counts.most_common(1)[0]
            exact = pattern([(mode>>j)&1 for j in range(7)]) if freq>=2 else 'Padrao variavel entre replicas'
            row.update(padrao_consenso_2de3=pattern(consensus), n_blocos_consenso=int(consensus.sum()),
                       padrao_exato_recorrente=exact, nrep_padrao_exato=int(freq),
                       n_blocos_3de3=int((active.sum(axis=0)==3).sum()))
            med=np.median(x,axis=0); other=np.delete(med,3)
            row['A_outros6_max_mediana']=float(other.max())
            row['contraste_mito_menos_max_outros']=float(med[3]-other.max())
            row['especificidade_operacional_sem_cosseno']=bool((active[:,3].sum()>=2) and not np.delete(active,3,axis=1).any())
        else:
            row.update(padrao_consenso_2de3='Nao avaliavel', n_blocos_consenso=np.nan,
                       padrao_exato_recorrente='Nao avaliavel', nrep_padrao_exato=0, n_blocos_3de3=np.nan,
                       A_outros6_max_mediana=np.nan, contraste_mito_menos_max_outros=np.nan,
                       especificidade_operacional_sem_cosseno=False)
        if 'contagem_relativa_DMSO' in g:
            vals=pd.to_numeric(g['contagem_relativa_DMSO'], errors='coerce')
            row['contagem_relativa_min_diagnostico']=float(vals.min()) if vals.notna().any() else np.nan
        rows.append(row); arrays.append(x)
    summary=pd.DataFrame(rows); cube=np.stack(arrays)
    origcols=KEY+['prioridade_exploratoria','A_Mito_mediana','cosseno_mito_min']
    for c in origcols:
        if c not in original: raise ValueError('Resumo original sem '+c)
    old=original[origcols].rename(columns={'A_Mito_mediana':'A_Mito_mediana_original',
                                         'prioridade_exploratoria':'prioridade_R1_R3_original'})
    summary=summary.merge(old,on=KEY,how='left',validate='one_to_one',indicator=True,sort=False)
    if not summary['_merge'].eq('both').all() or len(summary)!=len(original):
        raise ValueError('Conjunto de condicoes diverge do resumo original.')
    summary=summary.drop(columns='_merge')
    complete=summary['completa_3rep_7blocos'].to_numpy(bool)
    if not np.allclose(summary.loc[complete,'A_Mito_mediana'],summary.loc[complete,'A_Mito_mediana_original'],rtol=1e-9,atol=1e-10):
        raise ValueError('Medianas de Mito nao reproduzem os escores originais.')
    rule=summary['especificidade_operacional_sem_cosseno'] & (summary['cosseno_mito_min']>0)
    if not np.array_equal(rule,summary['prioridade_R1_R3_original'].to_numpy(bool)):
        raise ValueError('A regra original de desenvolvimento nao foi reproduzida.')
    if not complete.any(): raise ValueError('Nenhuma condicao completa para o mapa.')
    good=summary.loc[complete].reset_index(drop=True)
    x=cube[complete]; a=x>CUT; N=len(good)
    block_rows=[]
    for j,b in enumerate(BLOCKS):
        counts=a[:,:,j].sum(axis=1)
        block_rows.append(dict(bloco=b,n_condicoes_avaliaveis=N,n_acima_2de3=int((counts>=2).sum()),
                               n_acima_3de3=int((counts==3).sum()),pct_acima_2de3=100*np.mean(counts>=2),
                               pct_acima_3de3=100*np.mean(counts==3)))
    combos=[]
    for size in [2,3]:
        for ids in combinations(range(7),size):
            simultaneous=a[:,:,ids].all(axis=2).sum(axis=1)
            marginal=(a[:,:,ids].sum(axis=1)>=2).all(axis=1)
            combos.append(dict(ordem=size,combinacao=' + '.join(BLOCKS[j] for j in ids),
                               n_inclusiva_simultanea_2de3=int((simultaneous>=2).sum()),
                               pct_inclusiva_simultanea_2de3=100*np.mean(simultaneous>=2),
                               n_inclusiva_simultanea_3de3=int((simultaneous==3).sum()),
                               n_apenas_consenso_marginal=int(marginal.sum()),denominador=N))
    plate_rows=[]
    for plate in sorted(good[P].unique()):
        sel=good[P].eq(plate).to_numpy()
        for j,b in enumerate(BLOCKS):
            flag=a[sel,:,j].sum(axis=1)>=2
            plate_rows.append(dict(placa=plate,bloco=b,n_condicoes=int(sel.sum()),n_acima_2de3=int(flag.sum()),pct_acima_2de3=100*flag.mean()))
    corr_rows=[]
    for j,b in enumerate(BLOCKS):
        for r1,r2 in combinations(range(3),2):
            v=pd.DataFrame({'a':x[:,r1,j],'b':x[:,r2,j]})
            rho=v.corr(method='spearman').iloc[0,1]
            corr_rows.append(dict(bloco=b,par=REPS[r1]+' / '+REPS[r2],rho_spearman=rho,n_condicoes=N))
    patterns=good.groupby(['padrao_exato_recorrente'],sort=True).size().rename('n_condicoes').reset_index()
    patterns['pct_condicoes']=100*patterns.n_condicoes/N
    patterns=patterns.sort_values(['n_condicoes','padrao_exato_recorrente'],ascending=[False,True])
    return dict(todas=summary, completas=good, cube=x, blocos=pd.DataFrame(block_rows),
                combinacoes=pd.DataFrame(combos), placas=pd.DataFrame(plate_rows),
                correlacoes=pd.DataFrame(corr_rows), padroes=patterns)




In [ ]:
RESULT = analyze(rep, original)
print('Condicoes completas:', len(RESULT['completas']), '/', len(RESULT['todas']))
print('Pocos de tratamentos:', int(RESULT['todas']['n_pocos'].sum()))
print('Codigos de compostos:', RESULT['todas'][E].nunique())
print('Prioridades originais reproduzidas:', int(RESULT['todas']['prioridade_R1_R3_original'].sum()))
display(RESULT['blocos'])
display(RESULT['padroes'].head(15))
for size in [2,3]:
    display(RESULT['combinacoes'].query('ordem == @size').sort_values(
        ['n_inclusiva_simultanea_2de3','combinacao'],ascending=[False,True]).head(10))


## 4. Figuras e PCA exploratorias
Uma figura por grafico, com PNG de 300 dpi e SVG. O limite e fixo e os quadros completos sao exportados, mesmo quando a figura mostra apenas as combinacoes mais frequentes.

A PCA usa log1p das medianas por condicao e padronizacao por coluna. Ha uma representacao com AGP/DNA/ER/Mito e outra com os sete blocos. A segunda inclui tres descritores morfologicos: eles nao sao tres organelas adicionais. A PCA e ajustada a toda a base completa e nao mede desempenho preditivo. Cargas sao exportadas para interpretacao. Nao ha clustering automatico nem escolha de numero de grupos nesta etapa.

A correlacao de Spearman por bloco compara o ordenamento das condicoes entre replicas; nao substitui a concordancia das medidas dentro de cada assinatura.

In [ ]:
def pca_description(good, blocks):
    """PCA descritiva: log1p(A), padronizacao global, SVD; nao mede predicao."""
    raw=good[['A_'+b+'_mediana' for b in blocks]].to_numpy(float)
    if not np.isfinite(raw).all() or (raw<0).any(): raise ValueError('PCA sem imputacao: dados invalidos.')
    z=np.log1p(raw); mean=z.mean(0); std=z.std(0,ddof=0); keep=std>0
    if keep.sum()<2 or len(z)<3: return None
    X=(z[:,keep]-mean[keep])/std[keep]
    u,s,vt=np.linalg.svd(X,full_matrices=False)
    for i in range(len(s)):
        if vt[i,np.argmax(np.abs(vt[i]))]<0: vt[i]*=-1; u[:,i]*=-1
    coords=u*s; variance=s*s/np.sum(s*s)
    names=['PC'+str(i+1) for i in range(len(s))]
    table=good[KEY+[D,'prioridade_R1_R3_original','padrao_consenso_2de3']].copy()
    for i,c in enumerate(names): table[c]=coords[:,i]
    load=pd.DataFrame(vt.T,index=np.array(blocks)[keep],columns=names).reset_index(names='bloco')
    return dict(scores=table,loadings=load,variance=variance,
                parametros={'blocos':blocks,'media_log1p':mean.tolist(),'desvio_log1p':std.tolist(),
                            'blocos_constantes_excluidos':np.array(blocks)[~keep].tolist()})


def figures(result, out):
    """Graficos independentes; cores padrao do Matplotlib; sem paineis compostos."""
    dest=Path(out)/'figuras'; dest.mkdir(parents=True,exist_ok=True)
    captions=[]
    def save(fig,name,caption):
        fig.tight_layout()
        for ext in ['png','svg']: fig.savefig(dest/(name+'.'+ext),dpi=300,bbox_inches='tight')
        captions.append({'arquivo':name,'legenda':caption})
        plt.show(); plt.close(fig)
    b=result['blocos']; g=result['completas']; N=len(g)
    fig,ax=plt.subplots(figsize=(9,4.8)); y=np.arange(7)
    ax.barh(y-.16,b.n_acima_2de3,.3,label='Acima em pelo menos 2 replicas')
    ax.barh(y+.16,b.n_acima_3de3,.3,label='Acima nas 3 replicas')
    ax.set(yticks=y,yticklabels=LABELS,xlabel='Numero de condicoes',title='Excedencia dos limites por bloco em R1-R3')
    ax.legend(); save(fig,'B01_excedencia_blocos',f'N={N} condicoes completas. Limite A>1; contagens entre blocos se sobrepoem. Nao sao hits confirmados.')
    counts=g.n_blocos_consenso.value_counts().reindex(range(8),fill_value=0)
    fig,ax=plt.subplots(figsize=(8,4.5)); ax.bar(counts.index,counts.values)
    ax.set(xlabel='Numero de blocos acima em pelo menos 2 de 3 replicas',ylabel='Condicoes',xticks=range(8),title='Extensao multibloco: consenso marginal')
    save(fig,'B02_numero_blocos','Consenso marginal nao garante que os blocos excedam o limite juntos nas mesmas duas replicas. Zero nao significa ausencia de efeito.')
    patt=result['padroes'].head(15).iloc[::-1]
    fig,ax=plt.subplots(figsize=(11,max(5,.7*len(patt)))); ax.barh(range(len(patt)),patt.n_condicoes)
    ax.set_yticks(range(len(patt)),[textwrap.fill(str(v),46) for v in patt.padrao_exato_recorrente])
    ax.set(xlabel='Condicoes',title='Padrao exato dos sete blocos repetido em pelo menos 2 replicas')
    save(fig,'B03_padroes_exatos','Categorias mutuamente exclusivas. Padrao variavel: nenhuma configuracao completa dos sete blocos se repete em duas replicas. Tabela exportada contem todas as categorias.')
    combo=result['combinacoes']; mat=np.zeros((7,7),int)
    for j in range(7): mat[j,j]=int(b.iloc[j].n_acima_2de3)
    for j,k in combinations(range(7),2):
        name=BLOCKS[j]+' + '+BLOCKS[k]
        mat[j,k]=mat[k,j]=int(combo.loc[combo.combinacao.eq(name),'n_inclusiva_simultanea_2de3'].iloc[0])
    fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(mat)
    for j in range(7):
        for k in range(7): ax.text(k,j,str(mat[j,k]),ha='center',va='center')
    ax.set(xticks=range(7),xticklabels=LABELS,yticks=range(7),yticklabels=LABELS,title='Pares: excedencia conjunta nas mesmas duas replicas')
    plt.setp(ax.get_xticklabels(),rotation=45,ha='right'); fig.colorbar(im,ax=ax,label='Condicoes')
    save(fig,'B04_pares_simultaneos','Fora da diagonal: ambos os blocos A>1 nas mesmas pelo menos duas replicas. Inclusivo: outros blocos podem exceder tambem. Diagonal: excedencia individual 2/3. Nao mede sinergia.')
    tri=combo[combo.ordem.eq(3)].sort_values(['n_inclusiva_simultanea_2de3','combinacao'],ascending=[False,True]).head(10).iloc[::-1]
    fig,ax=plt.subplots(figsize=(11,7)); ax.barh(range(len(tri)),tri.n_inclusiva_simultanea_2de3)
    ax.set_yticks(range(len(tri)),[textwrap.fill(str(v),44) for v in tri.combinacao])
    ax.set(xlabel='Condicoes',title='Trios com maior frequencia de excedencia conjunta')
    save(fig,'B05_trios_simultaneos','Todos os tres blocos devem exceder juntos nas mesmas pelo menos duas replicas. Contagem inclusiva e descritiva; nao teste de interacao. Todas as 35 combinacoes constam da tabela.')
    fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(g.A_outros6_max_mediana,g.A_Mito_mediana,s=12,alpha=.4,label='Todas as condicoes completas')
    sel=g.prioridade_R1_R3_original.astype(bool)
    ax.scatter(g.loc[sel,'A_outros6_max_mediana'],g.loc[sel,'A_Mito_mediana'],s=70,marker='x',label='12 prioridades originais de R1-R3')
    ax.axhline(1,linestyle='--'); ax.axvline(1,linestyle=':')
    ax.set(xlabel='Maior mediana entre os outros seis blocos',ylabel='Mediana de A_Mito',title='Mito versus demais blocos: mapa descritivo')
    ax.legend(); save(fig,'B06_mito_demais_blocos','Medianas de R1-R3; contrastes nao sao testes de seletividade. As prioridades originais usaram tambem pior replica dos outros blocos e cosseno. R4 nao foi consultada.')
    plate=result['placas'].pivot(index='placa',columns='bloco',values='pct_acima_2de3').reindex(columns=BLOCKS)
    fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(plate.to_numpy(),aspect='auto')
    ax.set(xticks=range(7),xticklabels=LABELS,yticks=range(len(plate)),yticklabels=plate.index,title='Excedencia por placa e bloco')
    plt.setp(ax.get_xticklabels(),rotation=45,ha='right'); fig.colorbar(im,ax=ax,label='% de condicoes completas da placa')
    save(fig,'B07_placas','Denominador especifico por placa, exportado na tabela. Composicao de compostos e efeitos tecnicos nao sao separados por este grafico.')
    pc_results={}
    for tag,blocks in [('4canais',BLOCKS[:4]),('7blocos',BLOCKS)]:
        obj=pca_description(g,blocks)
        if obj is None: continue
        pc_results[tag]=obj
        fig,ax=plt.subplots(figsize=(8,5))
        for plate_name,grp in obj['scores'].groupby(P): ax.scatter(grp.PC1,grp.PC2,s=11,alpha=.5,label=plate_name)
        ax.set(xlabel=f'PC1 ({100*obj["variance"][0]:.1f}%)',ylabel=f'PC2 ({100*obj["variance"][1]:.1f}%)',title='PCA descritiva: '+tag)
        ax.legend(title='Placa',loc='best',fontsize=8)
        save(fig,'B08_PCA_'+tag,'PCA ajustada globalmente a log1p das medianas, com padronizacao por coluna. Exploracao, nao validacao nem classificador. Quatro canais e sete blocos sao representacoes distintas; morfologia nao equivale a organelas adicionais.')
    cor=result['correlacoes'].pivot(index='bloco',columns='par',values='rho_spearman').reindex(index=BLOCKS)
    fig,ax=plt.subplots(figsize=(7,5)); im=ax.imshow(cor.to_numpy(),vmin=-1,vmax=1,aspect='auto')
    ax.set(xticks=range(3),xticklabels=cor.columns,yticks=range(7),yticklabels=LABELS,title='Concordancia do ordenamento das condicoes entre replicas')
    fig.colorbar(im,ax=ax,label='Correlacao de Spearman')
    save(fig,'B09_correlacoes_replicas','Correlacao entre condicoes, por bloco e par de replicas; nao estima concordancia de cada assinatura individual. Nao foram calculados valores-p.')
    return captions,pc_results




## 5. Exportacoes e texto do TCC
O relatorio diferencia a nova descricao multibloco dos candidatos mitocondriais originais. O rascunho de texto e produzido somente com os valores efetivamente calculados.

**Modelagem supervisionada continua separada:** usar os mesmos escores para construir um rotulo e predize-lo produziria uma tarefa de reproducao da regra, nao uma demonstracao independente de mecanismo mitocondrial. O alvo cientifico precisa ser definido antes do treinamento; as particoes devem agrupar compostos e refazer etapas aprendidas. R4 ja foi consultada no projeto e nao sera reapresentada como intocada.

In [ ]:
def export_result(result,out,captions,pc_results,provenance):
    out=Path(out); tab=out/'tabelas'; tab.mkdir(exist_ok=True)
    for key,obj in result.items():
        if isinstance(obj,pd.DataFrame): obj.to_csv(tab/(key+'.csv'),index=False)
    for tag,obj in pc_results.items():
        obj['scores'].to_csv(tab/('PCA_'+tag+'_coordenadas.csv'),index=False)
        obj['loadings'].to_csv(tab/('PCA_'+tag+'_cargas.csv'),index=False)
        (out/('PCA_'+tag+'_parametros.json')).write_text(json.dumps(obj['parametros'],indent=2),encoding='utf-8')
    g=result['completas']; n=len(g); mito=g.nrep_Mito_acima.ge(2); comps=result['combinacoes']
    report=['# Mapa multibloco de R1-R3',
            'Analise descritiva pos-hoc. Sem leitura de R4, sem novos limiares e sem classificador supervisionado.',
            f'Condicoes de entrada: {len(result["todas"])}. Completas: {n}. Incompletas: {len(result["todas"])-n}.',
            f'Pocos de tratamentos representados: {int(result["todas"].n_pocos.sum())}. Codigos de compostos: {result["todas"][E].nunique()}.',
            f'Prioridades originais R1-R3 reproduzidas: {int(g.prioridade_R1_R3_original.sum())}.',
            '## Excedencia por bloco (A>1 em pelo menos duas replicas)']
    for row in result['blocos'].itertuples(): report.append(f'{row.bloco}: {row.n_acima_2de3}/{n} ({row.pct_acima_2de3:.2f}%); nas tres replicas: {row.n_acima_3de3}.')
    report+=['## Mito em contexto',f'{int(mito.sum())} condicoes excedem Mito em pelo menos 2/3 replicas. Isto nao aplica os demais criterios da selecao original.',
             '## Combinacoes simultaneas mais frequentes']
    for size in [2,3]:
        t=comps[comps.ordem.eq(size)].sort_values(['n_inclusiva_simultanea_2de3','combinacao'],ascending=[False,True]).head(5)
        for row in t.itertuples(): report.append(f'{row.combinacao}: {row.n_inclusiva_simultanea_2de3}/{n} condicoes (inclusivas).')
    report+=['## Limites e proximas decisoes',
             'O consenso marginal pode combinar blocos que nao excederam juntos nas mesmas duas replicas. A tabela de pares/trios exige simultaneidade.',
             'Blocos sao grupos de medidas, nao organelas mutuamente exclusivas. Nao ha bloco nucleolar independente.',
             'P95 e A>1 nao fornecem valores-p ou FDR. Escores relativos a referencias distintas nao sao unidades biologicas diretamente equivalentes.',
             'As PCA sao exploratorias e ajustadas aos dados completos. Nao sao uma avaliacao independente.',
             'Esses padroes operacionais nao devem virar automaticamente alvos de um classificador que use os mesmos escores.',
             'Para o objetivo supervisionado do TCC, ainda e necessario definir rotulos com proveniencia independente e particoes por composto; recalcular selecao/pre-processamento aprendido nos folds. R4 ja consultada nao sera um novo teste intocado.',
             'Nenhum resultado desta etapa substitui os 12 candidatos nem seus status originais.']
    text='\n\n'.join(report)+'\n'
    (out/'relatorio_multibloco.md').write_text(text,encoding='utf-8')
    legends='\n\n'.join('## '+c['arquivo']+'\n'+c['legenda'] for c in captions)
    (out/'legendas_figuras.md').write_text(legends,encoding='utf-8')
    draft=(f'Foram descritas {n} condicoes completas de R1-R3 a partir dos escores previamente calculados nos sete blocos. '
           'O mapa utilizou A>1 em pelo menos duas replicas, distinguindo o consenso marginal da excedencia conjunta de pares e trios nas mesmas replicas. '
           f'No bloco Mito, {int(mito.sum())} condicoes atenderam apenas ao criterio de magnitude 2/3, sem implicar seletividade ou confirmacao em R4. '
           'As combinacoes foram descritas sem testes de interacao ou controle de falsas descobertas. A PCA exploratoria foi examinada separadamente para quatro canais e sete blocos.\n')
    (out/'texto_TCC_multibloco_rascunho.md').write_text(draft,encoding='utf-8')
    provenance.update(n_condicoes=n,n_figuras=len(captions),R4_lida=False,modelo_supervisionado_treinado=False,
                      python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__)
    (out/'manifesto_multibloco.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
    return text


In [ ]:
CAPTIONS, PC_RESULTS = figures(RESULT, OUT)
TEXT = export_result(RESULT, OUT, CAPTIONS, PC_RESULTS, provenance)
# Confirma que nenhuma fonte mudou durante a execucao.
for k,path in FONTES.items():
    if sha256(path) != HASHES[k]: raise RuntimeError('Fonte alterada durante a execucao: '+str(path))
PACKAGE = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
display(Markdown(TEXT))
print(f'CONCLUIDO: {len(CAPTIONS)} figuras; tabelas, cargas de PCA, legendas e relatorio exportados.')
print('Pasta:', OUT)
print('Pacote:', PACKAGE)
print('R4 nao lida. Nenhum classificador treinado. Nenhum candidato original reclassificado.')


## Fontes tecnicas e rastreabilidade
Os valores e regras provem dos arquivos do proprio projeto, identificados por SHA-256 no manifesto. As definicoes computacionais novas estao documentadas neste notebook e sao descritivas.

[Medidas do CellProfiler 4.2.4](https://cellprofiler-manual.s3.amazonaws.com/CellProfiler-4.2.4/modules/measurement.html) - familias de medidas, objetos e canais nao equivalem a localizacao biologica exclusiva.

[Leitura de Parquet com Polars](https://docs.pola.rs/api/python/stable/reference/api/polars.read_parquet.html) - uso do leitor nativo; versao efetiva registrada.

**Limite de interpretacao:** a proximidade ou distancia no mapa e um resumo dos escores e nao confirma fissao, apoptose, disfuncao mitocondrial, equivalencia entre compartimentos ou mecanismo farmacologico.